# Get word embeddings

In [1]:
import os
import sys
import torch

import numpy as np
import pandas as pd

import torch.nn as nn

from tqdm import tqdm

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from metrics import EvaluationMetric
from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
train_dataset_path = os.path.join(base_data_path, 'classification_results/july_2026_results_2026-07-08/seed3/in_domain/spacy_large/x_y_train_set.csv')
train_df = DataProcessing.load_from_file(train_dataset_path)
train_df['Ground Truth'] = train_df['Ground Truth'].astype(int)
# train_df = train_df.sample(n=1000)
# train_df = train_df.loc[:1000, ]

test_dataset_path = os.path.join(base_data_path, 'classification_results/july_2026_results_2026-07-08/seed3/in_domain/spacy_large/x_y_test_set.csv')
test_df = DataProcessing.load_from_file(test_dataset_path)
test_df['Ground Truth'] = test_df['Ground Truth'].astype(int)
# test_df = test_df.sample(n=1000)
# test_df = test_df.loc[:1000, ]
test_df.head(3)

,Base Sentence,Ground Truth,Dataset Name,Base Sentence Embedding
0,Kobe Bryant's legacy will continue to inspire basketball players for decades.,0,chronicle2050,[-1.47754643e-02 1.25557244e-01 2.02721298e-01 -1.47769138e-01\n 2.77248830e-01 -5.39753437e-02 3.58213857e-02 8.86699855e-02\n 8.41418430e-02 2.09272337e+00 -1.97471321e-01 -4.02900912e-02\n 1.28088407e-02 1.42885074e-01 4.43354659e-02 -1.09935552e-03\n 1.01508126e-01 7.15602875e-01 -1.94591612e-01 5.83225898e-02\n -5.22811487e-02 -4.65752073e-02 3.34668644e-02 -1.68787166e-01\n 3.46106924e-02 1.88694075e-01 -1.84497729e-01 4.42585386e-02\n 7.88697824e-02 1.08215801e-01 -2.82965563e-02 -7.24094361e-02\n -1.73437327e-01 2.19445433e-02 1.15961075e-01 -2.11412176e-01\n 1.16339244e-01 -3.02162301e-02 8.23543817e-02 1.16972297e-01\n -1.27248913e-01 8.56716633e-02 1.41054377e-01 -3.27931494e-02\n -1.16470650e-01 -1.43213436e-01 -1.53985634e-01 1.96406454e-01\n -1....
1,"By 2030, it is predicted that the majority of new buildings will be constructed using sustainable and eco-friendly materials and designs.",1,chronicle2050,[-4.58022840e-02 1.60048053e-01 -1.41445994e-01 -1.72647417e-01\n -1.43633224e-02 1.10047854e-01 2.35692281e-02 5.71138412e-02\n -1.40190586e-01 2.03080440e+00 -1.82137609e-01 -4.29351814e-03\n -4.12109271e-02 2.43200990e-03 -6.36315718e-02 -1.05511598e-01\n -1.06377825e-02 1.41672897e+00 -1.41428128e-01 -7.03165308e-02\n -8.68011862e-02 1.29268207e-02 5.05953133e-02 -1.19209895e-02\n 7.61146173e-02 9.19433981e-02 -8.39005187e-02 1.15553020e-02\n -8.65483284e-02 3.68451513e-02 -4.14567031e-02 1.49557209e-02\n -5.81766590e-02 3.73929329e-02 7.03478456e-02 -1.52880490e-01\n 7.98651576e-02 6.76480010e-02 5.09174122e-03 -9.89307016e-02\n -3.44898179e-02 1.16678879e-01 5.73476180e-02 -3.17615941e-02\n -3.76515873e-02 -8.64548758e-02 -1.48876950e-01 -1.69498727e-01\n 4....
2,I think Gunnar is going to have a huge bounce back.,1,yt,[ 4.27430905e-02 2.05556586e-01 -2.25377038e-01 -4.50100005e-03\n 1.80477262e-01 -9.53214169e-02 -1.79280601e-02 -1.56816497e-01\n 8.46256688e-02 1.94842494e+00 -2.05941722e-01 2.25158525e-03\n 1.71609998e-01 4.84343618e-02 -1.55559912e-01 -6.10101372e-02\n -2.73834225e-02 9.15950775e-01 -2.01666519e-01 1.82439182e-02\n 9.17260051e-02 -5.24472408e-02 4.33084965e-02 -1.98448990e-02\n 1.84305105e-02 7.01507479e-02 -1.82103202e-01 -1.33276805e-01\n 6.36891648e-03 -9.58478227e-02 -9.29927453e-02 1.29952714e-01\n -8.22153315e-02 -4.22433438e-03 1.46869078e-01 -8.11135918e-02\n 8.61894116e-02 1.09462164e-01 -6.16556965e-02 -2.37056673e-01\n 1.86622497e-02 1.18374072e-01 9.28585709e-04 -6.14791699e-02\n 4.15291684e-03 1.54193625e-01 -1.77149937e-01 -1.77487925e-01\n -1....


## Tokenize -> Embed

- Before Tokenizing: Happy New Year!
- After Tokenizing: ['Happy', 'New', 'Year', '!']
- Pass each word one by one into embedding function

In [4]:
train_sfe = SpacyFeatureExtraction(train_df, 'Base Sentence', embedding_model_name="spacy_large")
train_embeddings_df = train_sfe.word_embeddings_extraction(reorder_cols=["Base Sentence", "Word", "Word Embedding", "Ground Truth"])
train_embeddings_df.head(3)

Embedding words: 100%|██████████| 119621/119621 [03:05<00:00, 643.20it/s]


,Base Sentence,Word,Word Embedding,Ground Truth,Dataset Name,Base Sentence Embedding
0,"Jim Laurie, ABC News, Hong Kong.",Jim,"[-0.44779, 0.35139, -0.053055, 0.037214, 0.23754, 0.10473, 0.14023, -0.24176, 0.050871, 0.58551, -0.76053, -0.8468, 0.16697, -0.28775, -0.15465, -0.34034, 0.034289, 0.51918, 0.11058, 0.12103, 0.12394, 0.17281, 0.37131, -0.15065, -0.16555, -0.22079, -0.85399, 0.25236, 0.06458, -0.00028618, -0.15626, 0.18516, -0.18667, 0.16821, -0.49444, 0.067304, 0.038776, 0.21444, -0.51691, 0.20923, 0.21859, -0.11612, -0.0011757, 0.32794, -0.15302, 0.54139, -0.0031888, -0.22496, 0.23868, 0.16908, 0.62782, 0.20371, 0.027479, -0.074188, -0.2783, 0.19931, -0.1898, 0.37851, 0.34481, -0.20809, -0.75139, 0.41441, 0.29857, 0.14671, 0.35206, -0.34111, -0.37011, 0.27775, -0.17551, -0.57679, 0.47325, -0.48732, 0.38127, -0.05154, 0.19056, -0.13794, -0.061313, 0.43138, -0.33547, 0.16933, -0.022904, 0.20129, 0.4400...",0,timebank,[-9.59910005e-02 4.25181925e-01 -1.00978911e-02 -1.97154999e-01\n 2.39281446e-01 1.37044609e-01 9.73707139e-02 -3.57897758e-01\n 1.14277892e-01 9.84569967e-01 -5.29778004e-01 -2.92470425e-01\n 1.14285789e-01 -7.50712752e-02 -1.74847782e-01 -8.65533575e-03\n -3.40668932e-02 7.49683261e-01 7.77822211e-02 2.11092547e-01\n 1.98684439e-01 3.05208206e-01 1.48093343e-01 -2.39156485e-01\n 6.01861142e-02 6.37311116e-02 -3.25881898e-01 7.97304511e-02\n 1.72715038e-01 -9.02096927e-02 -9.15940031e-02 -3.36566642e-02\n 6.82503358e-02 2.99961209e-01 -5.87911084e-02 3.48591171e-02\n 8.54506716e-03 1.43806890e-01 -2.55325109e-01 6.60302415e-02\n 4.47886437e-02 -2.39169329e-01 3.64966989e-02 1.96004584e-02\n -1.47448242e-01 9.90039930e-02 -1.23194307e-01 -2.14495331e-01\n 1....
1,"Jim Laurie, ABC News, Hong Kong.",Laurie,"[-0.3116, 0.72857, 0.21754, 0.17799, -0.05294, 0.17398, 0.011487, -0.51103, 0.12643, -0.33436, -0.60621, -0.89464, 0.17113, -0.34673, -0.25506, -0.026207, 0.057294, -0.22769, 0.2373, -0.26018, 0.65054, 0.11656, -0.14026, -0.45699, -0.1265, -0.027108, -0.41257, 0.25433, -0.26399, 0.35153, -0.13386, 0.17062, -0.03075, 0.33417, 0.15208, -0.10107, 0.23981, -0.15014, -0.38474, 0.24827, 0.37099, -0.12292, 0.11792, 0.0062123, 0.20695, 0.11691, 0.048537, -0.54061, 0.39799, 0.36725, 0.15885, 0.23836, -0.22311, -0.19606, -0.22052, -0.02105, 0.18056, 0.31031, 0.012409, -0.1296, -0.67055, 0.08641, -0.22088, -0.23909, -0.072092, 0.10597, -0.36558, -0.20614, -0.00014678, -0.16312, 0.24876, -0.11506, 0.28464, 0.39353, 0.26175, 0.27505, -0.13433, 0.32208, -0.21828, 0.35252, 0.28907, 0.35333, 0.032964,...",0,timebank,[-9.59910005e-02 4.25181925e-01 -1.00978911e-02 -1.97154999e-01\n 2.39281446e-01 1.37044609e-01 9.73707139e-02 -3.57897758e-01\n 1.14277892e-01 9.84569967e-01 -5.29778004e-01 -2.92470425e-01\n 1.14285789e-01 -7.50712752e-02 -1.74847782e-01 -8.65533575e-03\n -3.40668932e-02 7.49683261e-01 7.77822211e-02 2.11092547e-01\n 1.98684439e-01 3.05208206e-01 1.48093343e-01 -2.39156485e-01\n 6.01861142e-02 6.37311116e-02 -3.25881898e-01 7.97304511e-02\n 1.72715038e-01 -9.02096927e-02 -9.15940031e-02 -3.36566642e-02\n 6.82503358e-02 2.99961209e-01 -5.87911084e-02 3.48591171e-02\n 8.54506716e-03 1.43806890e-01 -2.55325109e-01 6.60302415e-02\n 4.47886437e-02 -2.39169329e-01 3.64966989e-02 1.96004584e-02\n -1.47448242e-01 9.90039930e-02 -1.23194307e-01 -2.14495331e-01\n 1....
2,"Jim Laurie, ABC News, Hong Kong.",",","[-0.082752, 0.67204, -0.14987, -0.064983, 0.056491, 0.40228, 0.0027747, -0.3311, -0.30691, 2.0817, 0.031819, 0.013643, 0.30265, 0.0071297, -0.5819, -0.2774, -0.062254, 1.1451, -0.24232, 0.1235, -0.12243, 0.33152, -0.006162, -0.30541, -0.13057, -0.054601, 0.037083, -0.070552, 0.5893, -0.30385, 0.2898, -0.14653, -0.27052, 0.37161, 0.32031, -0.29125, 0.0052483, -0.13212, -0.052736, 0.087349, -0.26668, -0.16897, 0.015162, -0.0083746, -0.14871, 0.23413, -0.20719, -0.091386, 0.40075, -0.17223, 0.18145, 0.37586, -0.28682, 0.37289, -0.16185, 0.18008, 0.3032, -0.13216, 0.18352, 0.095759, 0.094916, 0.008289, 0.11761, 0.3

In [5]:
test_sfe = SpacyFeatureExtraction(test_df, 'Base Sentence', embedding_model_name="spacy_large")
test_embeddings_df = test_sfe.word_embeddings_extraction(reorder_cols=["Base Sentence", "Word", "Word Embedding", "Ground Truth"])
test_embeddings_df.head(3)

Embedding words: 100%|██████████| 35711/35711 [00:55<00:00, 643.86it/s]


,Base Sentence,Word,Word Embedding,Ground Truth,Dataset Name,Base Sentence Embedding
0,Kobe Bryant's legacy will continue to inspire basketball players for decades.,Kobe,"[-0.78394, -0.45452, 0.919, -0.70767, 0.86838, 0.049567, 0.15573, -0.23261, 0.32929, -0.032237, -0.87793, 0.053554, -0.44069, 0.17965, 0.36076, -0.0039255, 0.86855, 0.24361, -0.24883, 0.17946, -0.29891, 0.083592, -0.11411, -0.815, -0.68318, 0.12481, -0.12571, -0.3887, 0.62788, -0.045558, -0.14818, 0.50745, -0.38633, -0.014417, -0.12861, -0.80414, 0.2955, -0.38698, -0.51201, 0.47827, -0.51145, -0.3007, -0.055202, 0.040058, -0.23991, -0.62669, -0.34964, 0.1106, -0.55021, 0.58441, -0.11376, 0.098474, -0.057006, -0.47301, -0.060898, -0.30087, -0.18004, 0.018862, -0.33081, 0.26041, -0.023596, -0.45176, -0.033196, -0.5115, 0.51409, -0.12231, 0.23581, 0.68169, 0.31765, -0.10448, -0.41085, -0.6909, 0.39327, 0.31033, -0.30804, -0.63137, 0.18444, 0.53899, 0.091017, 0.038676, 0.4125, 0.19056, -0....",0,chronicle2050,[-1.47754643e-02 1.25557244e-01 2.02721298e-01 -1.47769138e-01\n 2.77248830e-01 -5.39753437e-02 3.58213857e-02 8.86699855e-02\n 8.41418430e-02 2.09272337e+00 -1.97471321e-01 -4.02900912e-02\n 1.28088407e-02 1.42885074e-01 4.43354659e-02 -1.09935552e-03\n 1.01508126e-01 7.15602875e-01 -1.94591612e-01 5.83225898e-02\n -5.22811487e-02 -4.65752073e-02 3.34668644e-02 -1.68787166e-01\n 3.46106924e-02 1.88694075e-01 -1.84497729e-01 4.42585386e-02\n 7.88697824e-02 1.08215801e-01 -2.82965563e-02 -7.24094361e-02\n -1.73437327e-01 2.19445433e-02 1.15961075e-01 -2.11412176e-01\n 1.16339244e-01 -3.02162301e-02 8.23543817e-02 1.16972297e-01\n -1.27248913e-01 8.56716633e-02 1.41054377e-01 -3.27931494e-02\n -1.16470650e-01 -1.43213436e-01 -1.53985634e-01 1.96406454e-01\n -1....
1,Kobe Bryant's legacy will continue to inspire basketball players for decades.,Bryant,"[-0.60304, 0.38201, 0.96666, -0.16499, 0.81235, 0.21212, 0.010079, -0.034119, 0.14021, -0.24526, -0.26848, -0.27705, -0.24261, -0.27228, -0.44451, 0.10191, 0.60805, 0.061457, -0.14529, -0.071975, 0.14353, -0.029022, 0.23699, -0.54741, -0.33612, 0.13448, -0.47953, -0.17967, 0.12098, 0.62848, 0.12241, 0.25602, -0.46788, 0.20608, -0.18688, -0.41405, -0.23114, -0.13109, -0.19605, 0.01585, -0.24026, -0.062213, 0.018155, 0.41591, -0.19422, -0.038717, -0.42223, 0.019986, -0.27363, 0.49999, -0.11567, 0.30097, -0.37315, -0.22652, 0.23159, -0.66451, 0.12978, 0.31214, 0.38549, 0.2485, -0.029207, -0.27309, 0.099036, -0.43276, 0.45051, 0.16482, -0.016918, 0.24518, 0.33439, -0.12746, -0.33955, -0.82283, 0.19784, -0.081495, 0.13539, -0.039929, 0.20781, 0.8675, 0.33472, -0.47235, 0.28245, 0.31455, -0....",0,chronicle2050,[-1.47754643e-02 1.25557244e-01 2.02721298e-01 -1.47769138e-01\n 2.77248830e-01 -5.39753437e-02 3.58213857e-02 8.86699855e-02\n 8.41418430e-02 2.09272337e+00 -1.97471321e-01 -4.02900912e-02\n 1.28088407e-02 1.42885074e-01 4.43354659e-02 -1.09935552e-03\n 1.01508126e-01 7.15602875e-01 -1.94591612e-01 5.83225898e-02\n -5.22811487e-02 -4.65752073e-02 3.34668644e-02 -1.68787166e-01\n 3.46106924e-02 1.88694075e-01 -1.84497729e-01 4.42585386e-02\n 7.88697824e-02 1.08215801e-01 -2.82965563e-02 -7.24094361e-02\n -1.73437327e-01 2.19445433e-02 1.15961075e-01 -2.11412176e-01\n 1.16339244e-01 -3.02162301e-02 8.23543817e-02 1.16972297e-01\n -1.27248913e-01 8.56716633e-02 1.41054377e-01 -3.27931494e-02\n -1.16470650e-01 -1.43213436e-01 -1.53985634e-01 1.96406454e-01\n -1....
2,Kobe Bryant's legacy will continue to inspire basketball players for decades.,'s,"[-0.06858, 0.4647, 0.13214, 0.18599, -0.037015, 0.32988, 0.17865, -0.25977, -0.26022, 2.5728, -0.25867, -0.66095, 0.081984, 0.010321, -0.12223, 0.0094609, -0.088657, 0.58367, -0.017465, -0.35569, -0.10182, 0.061941, -0.14267, -0.40544, 0.29834, 0.10003, 0.035899, 0.2292, 0.30278, -0.18259, -0.0011042, 0.25792, -0.054132, 0.15748, 0.061311, -0.30055, 0.33732, 0.40023, 0.042472, -0.30014, 0.062963, 0.072134, 0.060897, -0.062527, 0.27505, -0.13527, -0.2171, 0.019315, 0.038683

In [6]:
print(f"Rows: {len(test_embeddings_df)}")
print(f"Unique sentences: {test_embeddings_df['Base Sentence'].nunique()}")
print(f"Empty Word column: {test_embeddings_df['Word'].isna().sum()}")
print(f"Zero embeddings: {(test_embeddings_df['Word Embedding'].apply(lambda x: np.all(x == 0))).sum()}")
print(test_embeddings_df[['Base Sentence', 'Word', 'Word Embedding']].head(10))

Rows: 35711
Unique sentences: 800
Empty Word column: 0
Zero embeddings: 547
                                                                   Base Sentence  \
0  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
1  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
2  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
3  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
4  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
5  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
6  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
7  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
8  Kobe Bryant's legacy will continue to inspire basketball players for decades.   
9  Kobe Bryant's legacy will continue to inspire basketball players for decades.   


In [7]:
# Check original test_df
print(f"Original test_df sentences: {len(test_df)}")
print(f"Unique sentences in original: {test_df['Base Sentence'].nunique()}")

# Check after tokenization
print(f"\nAfter tokenization:")
# print(f"Total rows in test_tokenized_df: {len(test_tokenized_df)}")
# print(f"Unique sentences: {test_tokenized_df['Base Sentence'].nunique()}")

# Check after embedding extraction
print(f"\nAfter embedding extraction:")
print(f"Total rows in test_embeddings_df: {len(test_embeddings_df)}")
print(f"Unique sentences: {test_embeddings_df['Base Sentence'].nunique()}")
print(f"Rows with NaN embeddings: {test_embeddings_df['Word Embedding'].isna().sum()}")

# Check which sentences are missing
original_sentences = set(test_df['Base Sentence'].unique())
processed_sentences = set(test_embeddings_df['Base Sentence'].unique())
missing_sentences = original_sentences - processed_sentences
print(f"\nMissing {len(missing_sentences)} sentences after processing")



Original test_df sentences: 800
Unique sentences in original: 800

After tokenization:

After embedding extraction:
Total rows in test_embeddings_df: 35711
Unique sentences: 800
Rows with NaN embeddings: 0

Missing 0 sentences after processing


In [8]:
words_to_sentence = []
words_embeddings_to_sentence = []

previous_sentence = None
previous_y = None
sentence_count = 0

sentence_texts = []
sentence_embeddings = []
sentence_labels = []

for row_idx, row_data in train_embeddings_df.iterrows():

    sentence = row_data['Base Sentence']
    word = row_data['Word']
    word_embedding = row_data['Word Embedding']
    y = row_data['Ground Truth']

    if previous_sentence is None:
        previous_sentence = sentence
        previous_y = y

    if sentence == previous_sentence:
        words_to_sentence.append(word)
        words_embeddings_to_sentence.append(word_embedding)

    else:
        # Save completed sentence
        completed_sentence = " ".join(words_to_sentence)
        sentence_texts.append(completed_sentence)

        if len(sentence_texts) <= 3:
            print(f"\nSentence {len(sentence_texts)}")
            print(f"Words: {words_to_sentence}")
            print(f"Rebuilt: {completed_sentence}")

        sentence_embeddings.append(
            np.mean(
                np.vstack(words_embeddings_to_sentence),
                axis=0
            )
        )

        sentence_labels.append(previous_y)

        # Reset for new sentence
        words_to_sentence = [word]
        words_embeddings_to_sentence = [word_embedding]
        previous_sentence = sentence
        previous_y = y

    sentence_count += 1

# Flush final sentence
if len(words_to_sentence) > 0:
    completed_sentence = " ".join(words_to_sentence)
    sentence_texts.append(completed_sentence)

    if len(sentence_texts) <= 3:
        print(f"\nSentence {len(sentence_texts)}")
        print(f"Words: {words_to_sentence}")
        print(f"Rebuilt: {completed_sentence}")

    sentence_embeddings.append(
        np.mean(
            np.vstack(words_embeddings_to_sentence),
            axis=0
        )
    )

    sentence_labels.append(previous_y)


Sentence 1
Words: ['Jim', 'Laurie', ',', 'ABC', 'News', ',', 'Hong', 'Kong', '.']
Rebuilt: Jim Laurie , ABC News , Hong Kong .

Sentence 2
Words: ['Her', 'work', 'at', 'NetApp', 'included', 'strategically', 'repositioning', 'the', 'brand', 'in', 'the', 'category', 'and', 'a', 'major', 'global', 'relaunch', '.']
Rebuilt: Her work at NetApp included strategically repositioning the brand in the category and a major global relaunch .

Sentence 3
Words: ['The', 'Islamic', 'Saudi', 'Academy', 'has', 'twelve', 'hundred', 'mostly', 'American', 'students', 'but', 'would', 'take', 'thirty', '-', 'five', 'hundred', 'if', 'it', 'had', 'the', 'room', '.']
Rebuilt: The Islamic Saudi Academy has twelve hundred mostly American students but would take thirty - five hundred if it had the room .


In [9]:
sentence_df = pd.DataFrame({
    'Base Sentence': sentence_texts,
    'Sentence Embedding': sentence_embeddings,
    'Ground Truth': sentence_labels
})

sentence_df.head()

,Base Sentence,Sentence Embedding,Ground Truth
0,"Jim Laurie , ABC News , Hong Kong .","[-0.095991, 0.42518193, -0.010097891, -0.197155, 0.23928145, 0.13704461, 0.097370714, -0.35789776, 0.11427789, 0.98456997, -0.529778, -0.29247043, 0.11428579, -0.075071275, -0.17484778, -0.008655336, -0.034066893, 0.74968326, 0.07778222, 0.21109255, 0.19868444, 0.3052082, 0.14809334, -0.23915648, 0.060186114, 0.06373111, -0.3258819, 0.07973045, 0.17271504, -0.09020969, -0.091594, -0.033656664, 0.068250336, 0.2999612, -0.05879111, 0.034859117, 0.008545067, 0.14380689, -0.2553251, 0.06603024, 0.044788644, -0.23916933, 0.0364967, 0.019600458, -0.14744824, 0.09900399, -0.12319431, -0.21449533, 0.19374722, -0.08527678, -0.034037553, 0.09233488, 0.006030998, -0.023805223, -0.18523961, 0.17622845, 0.15373956, 0.03510522, 0.09331596, -0.08269289, -0.31669757, -0.04601609, 0.18272524, -0.007131...",0
1,Her work at NetApp included strategically repositioning the brand in the category and a major global relaunch .,"[0.0763725, 0.07550406, 0.020500831, 0.050050672, 0.13141902, -0.009187961, -0.109680995, -0.015911825, 0.024443723, 1.8740041, -0.19425699, 0.0055982736, 0.08023339, -0.13886282, 0.18348712, -0.06780488, -0.069117, 1.0284796, -0.14380246, 0.034454275, 0.025776502, 0.11924795, -0.12062572, -0.03308399, 0.15086605, 0.022599556, -0.036778614, -0.018737469, -0.004976401, 0.16415703, 0.04615561, 0.056809265, 0.050816838, 0.11865728, 0.015527331, -0.11245772, -0.038611952, -0.010163569, 0.0723264, -0.06343673, -0.050607614, 0.16077271, -0.07097258, -0.099430986, -0.05912194, -0.06288299, -0.10551983, -0.022322403, -0.052949946, 0.030841062, 0.13479012, 0.08302995, -0.024504049, 0.067473345, -0.0541115, 0.025282161, -0.04040572, -0.15028207, 0.040445894, -0.1671831, -0.03191673, -0.076362215...",0
2,The Islamic Saudi Academy has twelve hundred mostly American students but would take thirty - five hundred if it had the room .,"[0.022225868, 0.070147954, 0.013483434, -0.08190997, 0.09196949, -0.17423782, -0.12349147, -0.047324393, 0.092171475, 2.3022676, -0.17563958, 0.12443386, 0.030862581, -0.069053866, -0.025681196, -0.067056224, -0.1521154, 0.97016466, -0.06781078, -0.05710447, -0.062104173, -0.046058364, 0.018563887, -0.0042254822, 0.00052790926, -0.09476373, -0.17605968, -0.06596796, 0.07982865, 0.11555684, 0.016377548, 0.21769543, 0.031164771, -0.03755627, -0.07989118, -0.010407745, -0.085738175, 0.080009334, -0.066547684, 0.025996696, 0.009101612, -0.02443586, 0.13862304, -0.15120167, 0.05138865, 0.010591514, -0.023522258, -0.042624485, 0.064286254, -0.0003055768, -0.2779562, 0.03841735, 0.001035047, -0.08985381, -0.05527287, -0.0073450906, 0.052592434, -0.04010216, 0.06818677, 0.017075514, -0.0268891...",0
3,The pandemic had been so draining that they wanted to scream .,"[-0.026740998, 0.10462583, -0.09584392, -0.16480385, 0.044503063, 0.013354893, 0.060344588, -0.04380696, -0.100609004, 2.3989332, -0.097544424, 0.042814996, 0.30532137, -0.12424997, -0.093095005, -0.061162945, -0.107309096, 0.8415449, -0.26501492, 0.016431125, 0.10481601, -0.045946907, 0.027643668, 0.019459253, -0.0905717, 0.06063464, -0.12935679, -0.08616685, 0.032693338, -0.2324365, -0.16013198, 0.06357675, 0.013993253, 0.015116249, 0.17353784, -0.13547419, -0.029623166, 0.07131028, -0.1000429, -0.024902748, 0.000511171, 0.112986885, 0.058989912, -0.16730034, 0.022158993, -0.027536085, -0.22758426, -0.03849268, 0.037501838, 0.03688425, -0.0906525, -0.013405499, -0.040294748, 0.06552532, 0.17376916, 0.09108367, 0.007480083, -0.09321227, 0.029323423, 0.01540433, -0.062281907, 0.0393573...",0
4,"Production levels have been agreed with producers a long time ago , so a fall in consumption will lead to losses .","[-0.09564421, 0.24475597, -0.03338686, 0.0073387297, -0.05182897, -0.08898702, -0.006712334, -0.09969015, -0.069398, 2.39295, -0.19718538, 0.023890225, 0.07155723, 0.034171265, -0.060484175, -0.138595, 0.040907405, 1.2070627, -0.26456517, 0.069674365, -0.03877627, -0.

In [10]:
X = np.vstack(sentence_df['Sentence Embedding'].values)
y = sentence_df['Ground Truth'].values

print(X.shape)
print(y.shape)

(2400, 300)
(2400,)


In [11]:
X = np.vstack(train_df['Base Sentence Embedding'].values)
y = train_df['Ground Truth'].values

print(X.shape)
print(y.shape)

(2400, 1)
(2400,)


In [12]:
# Average word embeddings per sentence using fast groupby
train_sentence_embeddings = dict(zip(
    train_embeddings_df.groupby('Base Sentence', sort=False).groups.keys(),
    train_embeddings_df.groupby('Base Sentence', sort=False)['Word Embedding'].apply(lambda x: np.mean(np.vstack(x), axis=0))
))
train_df['Word Averaged Embedding'] = train_df['Base Sentence'].map(train_sentence_embeddings)

test_sentence_embeddings = dict(zip(
    test_embeddings_df.groupby('Base Sentence', sort=False).groups.keys(),
    test_embeddings_df.groupby('Base Sentence', sort=False)['Word Embedding'].apply(lambda x: np.mean(np.vstack(x), axis=0))
))
test_df['Word Averaged Embedding'] = test_df['Base Sentence'].map(test_sentence_embeddings)

In [13]:
# Drop any missing embeds and stack into 2D arrays
train_df = train_df.dropna(subset=['Word Averaged Embedding'])
test_df = test_df.dropna(subset=['Word Averaged Embedding'])

X_train = np.vstack(train_df['Word Averaged Embedding'].values)
y_train = train_df['Ground Truth'].values

X_test = np.vstack(test_df['Word Averaged Embedding'].values)
y_test = test_df['Ground Truth'].values

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (2400, 300), y_train: (2400,)
X_test: (800, 300), y_test: (800,)


In [14]:
from classification_models import SkLearnModelFactory

ml_model_names = [
    'perceptron', 'sgd_classifier', 'logistic_regression', 'ridge_classifier',
    'decision_tree_classifier', 'random_forest_classifier',
    'gradient_boosting_classifier', 'support_vector_machine_classifier', 'x_gradient_boosting_classifier'
]

def build_models(factory, model_names, seed, reweight_class):
    """Initialize ML models from factory."""
    models = {}
    for name in model_names:
        models[name] = factory.select_model(name, random_state=seed, class_weight=reweight_class)
    return models

seed = 42
reweight_class = None
models = build_models(SkLearnModelFactory, ml_model_names, seed=seed, reweight_class=reweight_class)

results = []

for name, model in models.items():
    trained_model = model.train_model(X_train, y_train)
    preds = trained_model.predict(X_test)

    report = EvaluationMetric.eval_classification_report(y_test, preds)
    results.append({
        'Model': name,
        'Test Accuracy': report.get('accuracy', 0),
        'Precision (1)': report.get('1', {}).get('precision', 0),
        'Recall (1)': report.get('1', {}).get('recall', 0),
        'F1 (1)': report.get('1', {}).get('f1-score', 0)
    })

              precision    recall  f1-score   support

           0       0.87      0.88      0.88       405
           1       0.88      0.86      0.87       395

    accuracy                           0.87       800
   macro avg       0.87      0.87      0.87       800
weighted avg       0.87      0.87      0.87       800

              precision    recall  f1-score   support

           0       0.85      0.92      0.88       405
           1       0.91      0.84      0.87       395

    accuracy                           0.88       800
   macro avg       0.88      0.88      0.88       800
weighted avg       0.88      0.88      0.88       800

              precision    recall  f1-score   support

           0       0.86      0.90      0.88       405
           1       0.89      0.85      0.87       395

    accuracy                           0.88       800
   macro avg       0.88      0.88      0.88       800
weighted avg       0.88      0.88      0.88       800

              preci

/Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/.venv_predictions/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [16:53:29] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "probability" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.83      0.91      0.87       405
           1       0.90      0.81      0.85       395

    accuracy                           0.86       800
   macro avg       0.86      0.86      0.86       800
weighted avg       0.86      0.86      0.86       800

